# Diabetes Prediction using PySpark ML Pipeline
---
This Jupyter Notebook implements a distributed and scalable Machine Learning pipeline using **Apache PySpark**. We train and compare multiple classifiers (**Logistic Regression, Random Forest, GBT, LinearSVC**) tuned via **K-Fold Cross-Validation** on the Pima Indians Diabetes Dataset.

### 1. Import Dependencies and Initialize Spark Session
We build a PySpark session configured to run locally using all available CPU cores.

In [ ]:
import os
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Create local Spark Session
spark = SparkSession.builder \
    .appName("DiabetesPredictionNotebook") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized! Version:", spark.version)

### 2. Ingest and Inspect the Dataset
We load the CSV data, print its schema, and inspect the first few rows.

In [ ]:
data_path = "../data/diabetes.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)
df.show(5)
print("Row count:", df.count())
df.printSchema()

### 3. Exploratory Data Analysis & Handling Zero Values
In the Pima Indians dataset, features like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` have `0` values that represent missing entries. We replace these `0`s with `None` so that they can be handled by the Spark `Imputer` stage.

In [ ]:
impute_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Zero count before preprocessing:")
for col in impute_cols:
    zero_count = df.filter(F.col(col) == 0).count()
    print(f"  {col}: {zero_count} zeros")

# Replace 0s with Null/None
for col in impute_cols:
    df = df.withColumn(col, F.when(F.col(col) == 0, None).otherwise(F.col(col)))

print("\nZero count after replacing with Null:")
for col in impute_cols:
    null_count = df.filter(F.col(col).isNull()).count()
    print(f"  {col}: {null_count} nulls")

### 4. Splitting Train and Test Sets
We cache the dataframes to speed up computation during grid search.

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()
print(f"Training set: {train_df.count()} rows | Testing set: {test_df.count()} rows")

### 5. Building Preprocessing Stages
We construct the Pipeline components:
1. **Imputer**: Imputes Null values with the column medians.
2. **VectorAssembler**: Aggregates features into a single dense vector column.
3. **StandardScaler**: Centers and normalizes features.

In [ ]:
# 1. Median Imputation
output_impute_cols = [f"{c}_imputed" for c in impute_cols]
imputer = Imputer(inputCols=impute_cols, outputCols=output_impute_cols, strategy="median")

# 2. Feature Assembly
feature_cols = ["Pregnancies"] + output_impute_cols + ["DiabetesPedigreeFunction", "Age"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="assembled_features")

# 3. Scaling
scaler = StandardScaler(inputCol="assembled_features", outputCol="features", withStd=True, withMean=True)

### 6. Model Tuning and Benchmarking
We define a dictionary of classifiers and parameter grids, then run 3-Fold Cross-Validation on each.

In [ ]:
# Define models
lr = LogisticRegression(labelCol="Outcome", featuresCol="features")
rf = RandomForestClassifier(labelCol="Outcome", featuresCol="features", seed=42)
gbt = GBTClassifier(labelCol="Outcome", featuresCol="features", seed=42)
svc = LinearSVC(labelCol="Outcome", featuresCol="features")

models = {
    "LogisticRegression": (lr, 
                           ParamGridBuilder()\
                           .addGrid(lr.regParam, [0.01, 0.1])\
                           .build()),
    "RandomForest": (rf, 
                     ParamGridBuilder()\
                     .addGrid(rf.numTrees, [10, 30])\
                     .addGrid(rf.maxDepth, [5, 7])\
                     .build()),
    "GradientBoosting": (gbt, 
                         ParamGridBuilder()\
                         .addGrid(gbt.maxIter, [10, 20])\
                         .build()),
    "SupportVectorMachine": (svc, 
                             ParamGridBuilder()\
                             .addGrid(svc.regParam, [0.01, 0.1])\
                             .build())
}

# Evaluators
evaluator_roc = BinaryClassificationEvaluator(labelCol="Outcome", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="Outcome", predictionCol="prediction", metricName="accuracy")

results_summary = {}
for name, (classifier, paramGrid) in models.items():
    pipeline = Pipeline(stages=[imputer, assembler, scaler, classifier])
    
    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator_roc,
        numFolds=3,
        seed=42
    )
    
    print(f"Training {name} with grid-search CV...")
    cv_model = cv.fit(train_df)
    predictions = cv_model.transform(test_df)
    
    # Compute metrics
    roc_auc = evaluator_roc.evaluate(predictions)
    accuracy = evaluator_acc.evaluate(predictions)
    
    print(f"  -> Accuracy: {accuracy:.4f} | ROC-AUC: {roc_auc:.4f}\n")
    results_summary[name] = {"Accuracy": accuracy, "ROC-AUC": roc_auc}

### 7. Results Comparison Summary

In [ ]:
results_df = pd.DataFrame(results_summary).T
results_df

### 8. Stop Spark Session
Clean up resource allocations.

In [ ]:
spark.stop()
print("Spark Session terminated.")